# Optimización para el aprendizaje supervisado

**Departamento de Ciencias Físicas, Exactas & Energía / FIDCA**

<!--
Copyright © Angela Villota y Aníbal Sosa.

El contenido textual está licenciado bajo Creative Commons
Attribution-NonCommercial 4.0 International (CC BY-NC 4.0).

El código y la lógica de los notebooks están licenciados bajo
PolyForm Noncommercial License 1.0.0.

Consulte el archivo LICENSE del repositorio para conocer los términos completos.
-->

***

*Course:* [Math 535](https://people.math.wisc.edu/~roch/mmids/) - Mathematical Methods in Data Science (MMiDS)  
*Author:* [Sebastien Roch](https://people.math.wisc.edu/~roch/), Department of Mathematics, University of Wisconsin-Madison  
*Updated:* July 15, 2024   
*Copyright:* &copy; 2024 Sebastien Roch

***

Con este proyecto buscamos, explorar dentro del marco conceptual de las funciones de varias variables, la teoría de optimización matemática aplicada a un problema simplificado de inteligencia artificial.

En primer lugar, vamos a establecer condiciones básicas de optimalidad. En segundo lugar, estudiaremos un algoritmo de optimización básico: el método de descenso de gradiente, para encontrar el mínimo de una función continuamente diferenciable.

Finalmente implementaremos el algoritmo de gradiente descendente a un modelo de regresión logística, un método para modelar la probabilidad de un resultado binario basado en los atributos (características) de un conjunto de datos entrada. Todo lo anterior les permitirá aplicar  conceptos vistos en clase a un problema real de inteligencia artificial (aprendizaje automático).

In [ ]:
import numpy as np
from numpy import linalg as LA
import matplotlib.pyplot as plt
import pandas as pd

$\newcommand{\bmu}{\boldsymbol{\mu}}$
$\newcommand{\bSigma}{\boldsymbol{\Sigma}}$
$\newcommand{\bfbeta}{\boldsymbol{\beta}}$
$\newcommand{\bflambda}{\boldsymbol{\lambda}}$
$\newcommand{\bgamma}{\boldsymbol{\gamma}}$
$\newcommand{\bsigma}{{\boldsymbol{\sigma}}}$
$\newcommand{\bpi}{\boldsymbol{\pi}}$
$\newcommand{\btheta}{{\boldsymbol{\theta}}}$
$\newcommand{\bphi}{\boldsymbol{\phi}}$
$\newcommand{\balpha}{\boldsymbol{\alpha}}$
$\newcommand{\blambda}{\boldsymbol{\lambda}}$
$\renewcommand{\P}{\mathbb{P}}$
$\newcommand{\E}{\mathbb{E}}$
$\newcommand{\indep}{\perp\!\!\!\perp} \newcommand{\bx}{\mathbf{x}}$
$\newcommand{\bp}{\mathbf{p}}$
$\renewcommand{\bx}{\mathbf{x}}$
$\newcommand{\bX}{\mathbf{X}}$
$\newcommand{\by}{\mathbf{y}}$
$\newcommand{\bY}{\mathbf{Y}}$
$\newcommand{\bz}{\mathbf{z}}$
$\newcommand{\bZ}{\mathbf{Z}}$
$\newcommand{\bw}{\mathbf{w}}$
$\newcommand{\bW}{\mathbf{W}}$
$\newcommand{\bv}{\mathbf{v}}$
$\newcommand{\bV}{\mathbf{V}}$
$\newcommand{\bfg}{\mathbf{g}}$
$\newcommand{\bfh}{\mathbf{h}}$
$\newcommand{\horz}{\rule[.5ex]{2.5ex}{0.5pt}}$
$\renewcommand{\S}{\mathcal{S}}$
$\newcommand{\X}{\mathcal{X}}$
$\newcommand{\var}{\mathrm{Var}}$
$\newcommand{\pa}{\mathrm{pa}}$
$\newcommand{\Z}{\mathcal{Z}}$
$\newcommand{\bh}{\mathbf{h}}$
$\newcommand{\bb}{\mathbf{b}}$
$\newcommand{\bc}{\mathbf{c}}$
$\newcommand{\cE}{\mathcal{E}}$
$\newcommand{\cP}{\mathcal{P}}$
$\newcommand{\bbeta}{\boldsymbol{\beta}}$
$\newcommand{\bLambda}{\boldsymbol{\Lambda}}$
$\newcommand{\cov}{\mathrm{Cov}}$
$\newcommand{\bfk}{\mathbf{k}}$
$\newcommand{\idx}[1]{}$
$\newcommand{\xdi}{}$

## Parte I:
### Contexto: analizar la satisfacción del cliente


Citando [Wikipedia](https://en.wikipedia.org/wiki/Statistical_classification):

> En aprendizaje automático y estadística, la clasificación consiste en identificar a qué categoría (subpoblación) pertenece una nueva observación, basándose en un conjunto de datos de entrenamiento que contiene observaciones (o instancias) cuya categoría se conoce. Algunos ejemplos son la asignación de un correo electrónico a la clase "spam" o "no spam" y la asignación de un diagnóstico a un paciente según sus características observadas (sexo, presión arterial, presencia o ausencia de ciertos síntomas, etc.). La clasificación es un ejemplo de reconocimiento de patrones. En la terminología del aprendizaje automático, la clasificación se considera una instancia de aprendizaje supervisado, es decir, aprendizaje donde se dispone de un conjunto de entrenamiento de observaciones correctamente identificadas.

Ilustraremos este problema con un dataset [airline customer satisfaction](https://www.kaggle.com/datasets/sjleshrac/airlines-customer-satisfaction)  disponible en [Kaggle](https://www.kaggle.com), una excelente fuente de datos y análisis aportados por la comunidad. El contexto es el siguiente:

> El conjunto de datos contiene los datos de los clientes que ya han volado con una aerolínea. Se han consolidado los comentarios de los clientes sobre diversos contextos y sus datos de vuelo. El objetivo principal de este conjunto de datos es predecir si un futuro cliente estaría satisfecho o no con su servicio (problema de clasificación binaria), teniendo en cuenta  los valores asociados a todos los demás atributos (características) de un cliente.

Primero cargamos los datos (a continuación encuentran las opciones para hacerlo) y los convertimos a una representación matricial adecuada. Se ha hecho el análisis exploratorio de datos (o, más precisamente, ChatGPT)  ayudó a preprocesar el archivo original para eliminar las filas con datos faltantes o con cero calificaciones, convertir las variables categóricas en numéricas y conservar solo un subconjunto de las filas y columnas que usted va a utilizar. Puede ver los detalles del preprocesamiento en este [chat history](https://chatgpt.com/share/c5070b9c-f33f-4a37-a793-fde0d7cb7b06).

**El preprocesamiento es parte de lo que se conoce como Análisis Exploratorio de Datos (EDA), lo cual no es un requerimiento del proyecto mismo. Por esta razón, las siguientes líneas de código buscan dejar todo listo para que su trabajo se enfoque en el modelamiento matemático del problema.**

Sin embargo, siéntase en libertad de hacer su propio preprocesamiento o usar las herramientas que considere convenientes. Debe documentarlas claramente en cualquier caso.

A continuación usted encuentra tres formas distintas de cargar los datos o bien en Google Drive o bien en su máquina.

In [ ]:
from pathlib import Path

candidates = [
    Path("Invistico_Airline.csv"),
    Path("../data/Invistico_Airline.csv"),
]
data_path = next((path for path in candidates if path.exists()), None)
if data_path is None:
    data = None
    print("Dataset opcional no disponible. Descargue Invistico_Airline.csv y ubíquelo junto al notebook o en ../data/.")
else:
    data = pd.read_csv(data_path)
    display(data.head())

## Carga en Google Colab

Si trabaja en Google Colab, cargue `Invistico_Airline.csv` en el panel de archivos de la sesión y vuelva a ejecutar la celda anterior. No es necesario montar Google Drive ni usar rutas `/content/...`.

In [ ]:
if data is not None:
    display(data.head())
else:
    print("Cargue el dataset antes de continuar con el análisis.")

Este es un conjunto de datos extenso, más de cien mil registros (filas) y 23 atributos (columnas). Aquí están las primeras cinco filas y las primeras seis columnas.

In [ ]:
data.shape

In [ ]:
print(data.iloc[:5, :6])

Los nombres de las  columnas son:

In [ ]:
print(data.columns.tolist())

La primera columna indica si el cliente quedó satisfecho, lo cual puede modelarse definiendo dicha condición con el valor`1`. Las siguientes seis columnas ofrecen información sobre los clientes, como su edad o si pertenecen a un programa de fidelización de la aerolínea. En otras columnas los nombres  se explican por sí solos: `Type of Travel`, `Class`, etc.

Nuestro objetivo será predecir la primera columna, `satisfaction`, a partir del resto. Para ello, la retiramos de la matriz de datos $X$ y transformamos nuestros datos en matrices de Numpy.

In [ ]:
y = data['satisfaction'].to_numpy()
X = data.drop(columns=['satisfaction']).to_numpy()

## Análisis exploratorio básico

A continuación damos algunas ideas sobre el dataset. Esta sección puede omitirse.

Algunas características pueden afectar la satisfacción más que otras. Analicemos la edad, por ejemplo. El siguiente código extrae la columna "Edad" de "X" (es decir, la columna $3$) y calcula la proporción de clientes satisfechos en varios intervalos de edad.

Explicación de ChatGPT (autor del código):

1. [`numpy.digitize`](https://numpy.org/doc/stable/reference/generated/numpy.digitize.html) clasifica los datos de edad en los intervalos de edad especificados. El ajuste "-1" se utiliza para que coincida con la indexación basada en cero.

2. [`numpy.bincount`](https://numpy.org/doc/stable/reference/generated/numpy.bincount.html) cuenta las ocurrencias de cada índice de intervalo. El parámetro "minlength" garantiza que la longitud del array resultante coincida con el número de intervalos de edad (`age_labels`). Esto es importante si algunos contenedores tienen recuentos de cero, lo que garantiza que la matriz de recuentos cubra todos los contenedores.
3. `freq_satisfied = counts_satisfied / counts_all` calcula la frecuencia de satisfacción para cada grupo de edad dividiendo el número de clientes satisfechos por el número total de clientes en cada grupo de edad.

In [ ]:
age_col_index = 2
age_data = X[:, age_col_index]
age_bins = [0, 18, 25, 35, 45, 55, 65, 100]
age_labels = ['0-17', '18-24', '25-34', '35-44', '45-54', '55-64', '65+']
age_bin_indices = np.digitize(age_data, bins=age_bins) - 1
counts_all = np.bincount(age_bin_indices, minlength=len(age_labels))
counts_satisfied = np.bincount(age_bin_indices[y == 'satisfied'], minlength=len(age_labels))
freq_satisfied = counts_satisfied / counts_all
age_group_labels = np.array(age_labels)

Los resultados se grafican utilizando la función [`matplotlib.pyplot.bar`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.bar.html) de matplotlib. Observamos, en particular, que los jóvenes tienden a estar más insatisfechos. Claro que esto podría deberse a que no pueden permitirse los servicios más caros.

In [ ]:
plt.figure(figsize=(4, 4))
plt.bar(age_group_labels, freq_satisfied, color='lightblue', edgecolor='black')
plt.xlabel('Age Group'), plt.ylabel('Frequency of Satisfied Customers')
plt.show()

Podríamos seguir haciendo este tipo de análisis, pero eso se lo dejamos para un curso de IA. Ahora vamos a enfocarnos en la matemática aplicada al problema.

## Formulación matemática del problema general

Suponga que los datos de entrada tienen  la forma $\{(\boldsymbol{\alpha}_i, b_i) : i=1,\ldots, n\}$, donde $\boldsymbol{\alpha}_i \in \mathbb{R}^d$ son las características (atributos) y $b_i \in \{0,1\}$ es la etiqueta (`satisfied` o `dissatisfied`).

Nuestro objetivo es:

> Aprender un clasificador a partir de los ejemplos $\{(\boldsymbol{\alpha}_i, b_i) : i=1,\ldots, n\}$, es decir, una función de varias variables $\hat{f} : \mathbb{R}^d \to \mathbb{R}$ tal que $\hat{f}(\boldsymbol{\alpha}_i) \approx b_i$.

 Este problema se conoce como uno de [clasificación binaria](https://en.wikipedia.org/wiki/Binary_classification).

Un enfoque natural para este tipo de problema de [aprendizaje supervisado](https://en.wikipedia.org/wiki/Supervised_learning)$\idx{supervised learning}\xdi$ es definir dos objetos:

1. **Familia de clasificadores:** Una clase $\widehat{\mathbb{F}}$ de clasificadores (funciones de varias variables) de los cuales seleccionar $\hat{f}$.

2. **Función de pérdida:** Una función $\mathbb{L}(\hat{f}, (\mathbf{x},y))$ para cuantificar que tan bueno es el ajuste de $\hat{f}(\mathbf{x})$ con respecto a $y$.

Nuestro objetivo es entonces resolver

$$
\min_{\hat{f} \in \widehat{\mathbb{F}}} \frac{1}{n} \sum_{i=1}^n \mathbb{L}(\hat{f}, (\mathbf{x}_i, y_i)),
$$

es decir, buscamos un clasificador en $\widehat{\mathbb{F}}$ que minimice la pérdida promedio en los datos.

Para este proyecto utilizaremos un modelo de clasificación binaria conocido como *regresión logística* (https://en.wikipedia.org/wiki/Logistic_regression). Buscamos una función de los atributos (características) que calcule de forma aproximada la probabilidad de la etiqueta `satisfaction`. Para ello, modelamos el logaritmo de las probabilidades (o función logit) de la probabilidad de la etiqueta `satisfaction` como una función lineal de los atributos. Es decir, para este proyecto consideramos clasificadores no lineales $\hat{f}$ de la forma:

$$
\hat{f}(\mathbf{x})
= \sigma(\mathbf{x}^T \boldsymbol{\theta})
\qquad
\text{donde}
\qquad
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

y $\boldsymbol{\theta} \in \mathbb{R}^d$ es un vector de parámetros.  Esta función es la que define el modelo de regresión logística que mencionamos antes.

A continuación graficamos la función sigmoide para que usted observe su comportamiento.

In [ ]:
def sigmoid(z):
    return 1/(1+np.exp(-z))

grid = np.linspace(-5, 5, 100)
plt.plot(grid, sigmoid(grid), c='k')
plt.show()

Finalmente vamos a considerar la siguiente función de pérdida conocida como función de [pérdida de entropía cruzada](https://en.wikipedia.org/wiki/Cross_entropy#Cross-entropy_loss_function_and_logistic_regression)

$$
\mathbb{L}(\hat{f}, (\mathbf{x}, y))
= - \left(y \log(\sigma(\mathbf{x}^T \boldsymbol{\theta}))
+ (1-y) \log(1- \sigma(\mathbf{x}^T \boldsymbol{\theta}))\right).
$$

Podríamos usar otro clasificador $\hat{f}$ u otra función de pérdida, pero en este caso estas serán las funciones para trabajar.

## Parte II:
## Formulación del problema de optimización y las condiciones de optimalidad

En términos de nuestro problema, queremos resolver el problema de minimización:
$$
\min_{\boldsymbol{x} \in \mathbb{R}^d} \mathbb{L}(\mathbf{x};A,b)=\min_{\boldsymbol{x} \in \mathbb{R}^d}
- \frac{1}{n} \sum_{i=1}^n  \left(b_i\log(\sigma(\boldsymbol{\alpha}_i^T\mathbf{x} ))
+ \sum_{i=1}^n (1-b_i) \log(1- \sigma(\boldsymbol{\alpha}_i^T\mathbf{x}))\right).
$$

Para ello debe tener en cuenta que para obtener una predicción en $\{0,1\}$, podríamos seleccionar $\hat{f}(\mathbf{x})$ de acuerdo a un umbral $\tau \in [0,1]$, es decir, devolver un $1$ si $\hat{f}(\mathbf{x})> \tau$, esto es, $\mathbf{1}\{\hat{f}(\mathbf{x}) > \tau\}$, lo cual es  equivalente a la etiqueta `satisfied`, cuando se supera el umbral.

Aquí, implícitamente, se asume que el logaritmo es una función estrictamente creciente y, por lo tanto, no altera el máximo global de la función. Note que en la definición original de la función de pérdida $\mathbb{L}$, el multiplicar por $-1$ cambia el máximo global en un mínimo global.

Ahora bien, para resolver el problema de minimización vamos a implementar un algoritmo clásico de optimización: el gradiente descendente que explicamos a continuación.

**Algoritmo de gradiente descendente:**

Citando [Wikipedia](https://en.wikipedia.org/wiki/Gradient_descent):

> El descenso de gradiente es un algoritmo de optimización iterativo utilizado en aprendizaje automático para hallar el mínimo de una función (a menudo una función de pérdida). Funciona actualizando iterativamente los parámetros en la dirección opuesta del gradiente de la función, $- \nabla f$ con el objetivo de minimizar el error entre los resultados predichos y los reales.

En cada iteración del descenso del gradiente, damos un paso en la dirección del gradiente negativo, es decir,

$$
\mathbf{x}^{t+1}
= \mathbf{x}^t - \beta_t \nabla f(\mathbf{x}^t),
\quad t=0,1,2\ldots
$$

para una secuencia de pasos $\beta_t > 0$. Elegir el paso correcto (también conocido como longitud de paso o tasa de aprendizaje) es un tema complejo. Aquí solo consideraremos el caso de pasos fijos.

## Tarea 1. Gradiente de la función de pérdida $\mathbb{L}$

### 1.1

Para usar el descenso de gradiente, necesitamos el gradiente de $\mathbb{L}$ .  Utilice la regla de la cadena en una variable y pruebe que la derivada de $\sigma$,  es
$$
\sigma'(z) = \sigma(z)(1-\sigma(z))
$$

Esta expresión se conoce como ecuación diferencial logística. Surge en diversas aplicaciones, incluyendo el modelado de dinámica poblacional. En este caso, será una forma conveniente de calcular el gradiente.

### 1.2

Use el resultado anterior para mostrar que el gradiente

$$
\nabla\sigma(\mathbf{\alpha}^\top\mathbf{x})=\sigma'(\mathbf{\alpha}^\top\mathbf{x})\mathbf{\alpha}
$$

utilizando la regla de la cadena con respecto a $\mathbf{\alpha}=(\alpha_1,...,\alpha_d) \in \mathbb{R}^d$.

### 1.3

Debe mostrar que al calcular el gradiente de la función de pérdida $\mathbb{L}$ se obtiene:

$$
\nabla \mathbb{L}(\mathbf{x})=\frac{1}{n}\left(-\sum_{i=1}^n \frac{b_i}{\sigma(\alpha_i^\top\mathbf{x})}\nabla\sigma(\alpha_i^\top\mathbf{x})+\sum_{i=1}^n \frac{1-b_i}{1-\sigma(\alpha_i^\top\mathbf{x})}\nabla\sigma(\alpha_i^\top\mathbf{x})\right)
$$

o equivalentemente

$$
\nabla \mathbb{L}(\mathbf{x})=-\frac{1}{n}\sum_{i=1}^n (b_i-\sigma(\alpha_i^\top\mathbf{x}))\mathbf{\alpha_i}
$$



### 1.4

Finalmente, utilizando este último resultado muestre que es posible representar el gradiente de la función de pérdida en forma matricial como:

$$
\nabla \mathbb{L}(\mathbf{x};A,b)=-\frac{1}{n}\sum_{i=1}^n (b_i-\sigma(\mathbf{\alpha}_i^\top\mathbf{x}))\mathbf{\alpha}_i=-\frac{1}{n}A^\top\left[\mathbf{b}-\sigma(A\mathbf{x})\right].
$$

Considerando la representación matricial $A \in \mathbb{R}^{n \times d}$ con las filas $\boldsymbol{\alpha}_i^\top$, $i = 1,\ldots, n$ y $\boldsymbol{b} = (b_1, \ldots, b_n)^T \in \{0,1\}^n$.

## Tarea 2.  Algoritmo de gradiente descendente

### 2.1
Ahora vamos a implementar el algoritmo de gradiente descendente en Python. Asuma que una función `f` y su gradiente `grad_f` son dadas. Primero codifique la actualización del paso de descenso con tamaño de paso $\beta =$ `beta`.

In [ ]:
def desc_update(grad_f, x, beta): #Actualización del descenso
    return #Sú código aquí

def gd(f, grad_f, x0, beta=1e-3, niters=int(1e6)): # Descenso de gradiente de f

    #Sú código aquí

    return xk, f(xk)

### 2.2

Utilice su implementación para obtener el mínimo ($x=1$) de la función $f(x)=(x-1)^2+10$, definida junto con su derivada a continuación.

In [ ]:
def f(x):
    return (x-1)**2 + 10

def grad_f(x):
    return 2*(x-1)

xgrid = np.linspace(-5,5,100)
plt.plot(xgrid, f(xgrid), label='f')
plt.plot(xgrid, grad_f(xgrid), label='grad_f')
plt.ylim((-20,50)), plt.legend()
plt.show()

In [ ]:
gd(f, grad_f, 0)

### 2.3

El siguiente ejemplo muestra que se puede alcanzar un extremo local diferente dependiendo del punto de partida.

i. Usted debe identificar los tres puntos extremos y hallar los mínimos locales. ¿Hay un mínimo absoluto? ¿Hay un máximo absoluto?

ii. Utilice al menos tres diferentes valores para el paso  $\beta =$ `beta`. ¿Cambiar el tamaño del paso afecta el resultado? Explique su respuesta

iii. Utilice un applet de GeoGebra o una animación en Python para intentar ilustrar el punto anterior.

In [ ]:
def f(x):
    return 4 * (x-1)**2 * (x+1)**2 - 2*(x-1)

def grad_f(x):
    return 8 * (x-1) * (x+1)**2 + 8 * (x-1)**2 * (x+1) - 2

xgrid = np.linspace(-2,2,100)
plt.plot(xgrid, f(xgrid), label='f')
plt.ylim((-10,10)), plt.legend()
plt.show()

### 2.4

En el mundo real las funciones a optimizar suelen ser mucho mas complejas. Un ejemplo clásico es la función de Rosenbrock, que aquí presentamos en dos variables:

$$
f(x_1, x_2) = (a - x_1)^2 + b(x_2 - x_1^2)^2
$$

donde típicamente $a = 1$ y  $ b = 100 $. En este caso el único mínimo de ésta función es el punto $(1,1)$. Modifique su implementación del algoritmo de gradiente descendente para esta función (la versión anterior es para una función de valor real). Debe hallar el único mínimo partiendo de los siguientes puntos iniciales: $(-1.5,2.5)$, $(-5,5)$ y $(-15,5)$. Registre que observa y las razones por las cuales probablmente  no siempre se puede hallar el mínimo.

## Parte III

## Tarea 3. Modelo de regresión logística para clasificación

### 3.1

Para esta tarea deberá implementar el algoritmo de descenso de gradiente que propuso  en el punto anterior pero tomando un conjunto de datos como entrada. Recordemos que, para ejecutar el descenso de gradiente, primero implementamos una función que calcula una actualización de la dirección de descenso `desc_dir`. Debe construir esta función tomando como entrada la función `grad_fn` que calcula el gradiente, así como la iteración actual y el tamaño del paso `beta`. También debe introducir un conjunto de datos $A$ y $b$ como entrada adicional, siguiendo el siguiente esquema:

In [ ]:
def desc_dir(grad_fn, A, b, curr_x, beta):
    # sú código aquí
    return

### 3.2

Debe ahora implementar las funciones `loss_fn` y `grad_fn`,  usando la definición de la función sigmoide dada en la parte I. Debe definir `pred_fn` como $\bsigma(A \mathbf{x})$ y la función de pérdida

\begin{align*}
\mathbb{L}(\mathbf{x}; A, \mathbf{b})
&= -\frac{1}{n} \sum_{i=1}^n \left\{ b_i \log(\sigma(\boldsymbol{\alpha_i}^T \mathbf{x}))
+ (1-b_i) \log(1- \sigma(\boldsymbol{\alpha_i}^T \mathbf{x}))\right\}\
\end{align*}



In [ ]:
def pred_fn(x, A):
    return # sú código aquí

def loss_fn(x, A, b):
    return # sú código aquí

def grad_fn(x, A, b):
    return # sú código aquí

El algoritmo de gradiente descendente  tomará como entrada la función objetivo  `loss_fn` del problema de optimización, la función `grad_fn` que calcula el gradiente, el conjunto de datos `A`, el vector de etiquetas `b`, y un valor inicial `init_x`. Puede usar o no los parámetros opcionales: tamaño del paso y  número de iteraciones.

In [ ]:
def gd_for_logreg(loss_fn, grad_fn, A, b, init_x, beta=1e-3, niters=int(1e5)):
    curr_x = init_x

    # sú código aquí
    return curr_x

### Analizar la satisfacción del cliente

Ahora vamos a aplicar el modelo de regresión logística para estudiar la satisfacción de un pasajero con una aerolínea. Primero volvamos a ver los nombres de los atributos del conjunto de datos

In [ ]:
column_names = data.columns.tolist()
print(column_names)

Recuerde que nuestro objetivo es predecir la primera columna, `satisfaction`, a partir del resto. Para ello, transformamos nuestros datos en matrices de Numpy. También estandarizamos las columnas restando su media y dividiendo entre su desviación estándar (tal como lo hizo en la segunda jornada de resolución de problemas). Añadimos una columna de unos para tener en cuenta el término independiente de la regresión lineal.

In [ ]:
# Convertir las columnas (excepto 'satisfaction') a numérica
data_ = data.drop(columns=['satisfaction'])

# Convertir objetivo ('satisfaction') a numérico
# Mapear 'satisfied' -> 1 y 'dissatisfied' -> 0
data['satisfaction'] = data['satisfaction'].map({'satisfied': 1, 'dissatisfied': 0})

# convertir a numpy
y = data['satisfaction'].to_numpy()

# Use sólo columnas numericas
df_numeric = data_.select_dtypes(include=[np.number])
df_numeric = df_numeric.fillna(df_numeric.mean())

# Convertir a numpy
X = df_numeric.to_numpy()

# Calcular mean y std
mean = np.mean(X, axis=0)

std = np.std(X, axis=0)

# Standardize
X_standardized = (X - mean) / std# Add bias term (column of 1s)

A = np.concatenate((np.ones((len(y), 1)), X_standardized), axis=1)
b = y

### 3.3

Finalmente implemente el algoritmo de gradiente descendente diseñado para un problema de regresión logística general. El resultado final  equivale a encontrar los mejores parámetros para el modelo de clasificación de satisfacción de los pasajeros.

Explique la relación entre la solución del problema de optimización y la predicción del nivel de satisfacción de un pasajero.

In [ ]:
import time

# Assumiendo que A, b, loss_fn, grad_fn, y gd_for_logreg están  definidos
init_x = np.zeros(A.shape[1])

# Iniciar temporizador
start_time = time.time()

# Llamado al descenso de gradiente
best_x = gd_for_logreg(loss_fn, grad_fn, A, b, init_x, beta=1e-3, niters=int(1e3))

# Detener y calcular el tiempo de ejecución
elapsed_time = time.time() - start_time

print("Solución (best_x):", best_x)
print(f"Tiempo de ejecución: {elapsed_time:.4f} seconds")

### 3.4

Debe predecir si dos nuevos pasajeros, cuyos vectores de atributos (características)  son $\boldsymbol{\alpha}_{new}^1=$[1, 30, 204, 0, 1, 1, 2, 5, 1, 5, 5, 1, 1, 2, 4, 3, 5, 18, 30], y $\boldsymbol{\alpha}_{new}^2=$[1, 30, 504, 0, 1, 1, 2, 5, 1, 5, 5, 1, 1, 2, 4, 3, 5, 18, 15] estarán satisfechos o no, utilizando la función de predicción $p(\mathbf{x}; \boldsymbol{\alpha}) = \sigma(\boldsymbol{\alpha}^T \mathbf{x})$, donde $\mathbf{x}$ son los coeficientes ajustados, es decir, la solución del problema de minimización y $p$ es la probabilidad de estar satisfecho o no.

Debe suponer que se predice que un cliente estará satisfecho si $\sigma(\boldsymbol{\alpha}^T \mathbf{x}) > 0.5$.

In [ ]:
new_α1 = [1, 30, 204, 0, 1, 1, 2, 5, 1, 5, 5, 1, 1, 2, 4, 3, 5, 18, 30]
#new_α2 = [1, 30, 504, 0, 1, 1, 2, 5, 1, 5, 5, 1, 1, 2, 4, 3, 5, 18, 15]

## 3.5

 Encuentre una forma de calcular la exactitud de su modelo en el conjunto de datos dado. Es decir, calcule cuantos aciertos tuvo su predicción $\hat{f}$ respecto a los valores reales $\mathbf{b}$.